In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from scipy.spatial.distance import pdist
from sklearn.preprocessing import OrdinalEncoder


1. EDA and Data Preparation

In [ ]:
# Data import
path = Path("Data") / "Dataset1_BankClients.xlsx" # general path

data = pd.read_excel(path)

In [ ]:
# Check if there are missing values
data.isna().sum()

# Data description
data.describe()

In [ ]:
data.info()
print()

In [ ]:
print(f'Numero duplicati: {data.duplicated().sum()}')
print('Valori mancanti per colonna:\n', data.isna().sum())

In [ ]:
# Scalers
mm_scaler = MinMaxScaler() # min-max scaler
z_scaler = StandardScaler() # z scaler

In [ ]:
categorical_columns = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']
numerical_columns = ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']

# Split
numerical_features = data[numerical_columns]
categorical_features = data[categorical_columns].astype('category')

# Scale numerical
X_num = mm_scaler.fit_transform(numerical_features)

# Categorical representations
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
X_cat_encoded = encoder.fit_transform(categorical_features)

X_cat_kproto = categorical_features.astype(str).to_numpy()

ord_encoder = OrdinalEncoder()
X_cat_int = ord_encoder.fit_transform(categorical_features)

# Combined datasets
X_kproto = np.hstack((X_num, X_cat_kproto))
X_encoded = np.hstack((X_num, X_cat_encoded))
X_int = np.hstack((X_num, X_cat_int))

In [ ]:
# Histograms of categorical variables
for col in categorical_columns:
    counts = data[col].value_counts()

    plt.figure()
    counts.plot(kind="bar")

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
# Histograms and distributions of numerical variables

for col in numerical_columns:
    sns.histplot(data[col].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid()

    plt.tight_layout()
    plt.show()


In [ ]:
# Boxplots of numerical variabels
for col in numerical_columns:
    sns.boxplot(x=data[col], color='lightgreen')
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

In [ ]:
# Histograms, distributions and boxplots of numerical variables
for col in numerical_columns:
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    sns.histplot(data[col].dropna(), bins=30, kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")
    axes[0].grid()

    # Boxplot
    sns.boxplot(x=data[col], color='lightgreen', ax=axes[1])
    axes[1].set_title(f"Boxplot of {col}")
    axes[1].set_xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
sns.pairplot(data[numerical_columns])

In [ ]:
for col in categorical_columns:
    print(data[col].value_counts(normalize=True).head())

In [ ]:
distances = pdist(data[numerical_columns], metric="euclidean")

sns.histplot(distances, bins=50)
plt.title("Distribuzione delle distanze")

In [ ]:
# Correlation matrix
corr = numerical_features.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Numerical variables correlation matrix")
plt.show()

In [ ]:
abs_corr = corr.abs()
upper = abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
result = upper.stack().sort_values(ascending=False)
print(result)

2. Distances


2.1. K-Prototypes

$d(x,y) = \sum (x_i-y_i)^2 + \gamma \sum \delta(x_j,y_j)$,\
where $\delta=1$ if $x_j {=}\mathllap{/} y_j$ and $\delta=0$ if $x_j = y_j$

numericals -> distanza euclidea \
categoricals -> matching

In [107]:
from kmodes.kprototypes import KPrototypes

kproto = KPrototypes(
    n_clusters=3, # k=3 gives the best silhouette score
    init="Cao",
    random_state=42
)

cat_indices = list(range(len(numerical_columns),
                         len(numerical_columns) + len(categorical_columns)))

clusters = kproto.fit_predict(
    X_kproto,
    categorical=cat_indices
)

In [ ]:
data["cluster"] = clusters
data.groupby("cluster")[numerical_columns].mean()

In [ ]:
for col in categorical_columns:
    print(data.groupby("cluster")[col].value_counts(normalize=True))

In [109]:
def kproto_distance(x_num, x_cat, y_num, y_cat, gamma=1.0):
    num_dist = np.sum((x_num - y_num) ** 2)
    cat_dist = np.sum(x_cat != y_cat)
    return num_dist + gamma * cat_dist

def compute_kproto_distance_matrix(X_num, X_cat, gamma=1.0):
    n = X_num.shape[0]
    D = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):
            d = kproto_distance(X_num[i], X_cat[i], X_num[j], X_cat[j], gamma)
            D[i, j] = d
            D[j, i] = d

    return D

from sklearn.metrics import silhouette_score

D = compute_kproto_distance_matrix(X_num, X_cat_kproto, gamma=1.0)

score = silhouette_score(D, clusters, metric="precomputed")
print("Silhouette score:", score)

Silhouette score: 0.1715514673276773


In [ ]:
from sklearn.metrics import davies_bouldin_score

db_score = davies_bouldin_score(X_mm,clusters)
print(db_score)

2.2. Personalized Mixed Distance

$d(x,y) = \alpha d_{num} + (1-\alpha)d_{cat}$

In [ ]:
categorical_codes = categorical_features.apply(lambda col: col.cat.codes).to_numpy()

In [121]:
# Hamming distance for categorical variabels
from scipy.spatial.distance import pdist, squareform

def hamming_dstance_matrix(X_cat): # Put X_cat_int as input
    X_cat = np.asarray(X_cat)
    D = squareform(pdist(X_cat, metric='hamming'))
    return D

D_cat_hamming = hamming_dstance_matrix(X_cat_int)
D_cat_hamming

array([[0. , 0.6, 0.8, ..., 0.8, 0.8, 0.6],
       [0.6, 0. , 0.6, ..., 0.2, 0.6, 0.6],
       [0.8, 0.6, 0. , ..., 0.4, 0. , 0.4],
       ...,
       [0.8, 0.2, 0.4, ..., 0. , 0.4, 0.4],
       [0.8, 0.6, 0. , ..., 0.4, 0. , 0.4],
       [0.6, 0.6, 0.4, ..., 0.4, 0.4, 0. ]])

In [120]:
# Tanimoto/Jaccard distance for categorical variables

def tanimoto_distance_matrix(X_cat): # Put X_cat_encoded as input
    X_cat = np.asarray(X_cat)
    D = squareform(pdist(X_cat, metric='jaccard'))
    return D

D_cat_tanimoto = tanimoto_distance_matrix(X_cat_encoded)
D_cat_tanimoto

array([[0.        , 0.66666667, 0.8       , ..., 0.83333333, 0.8       ,
        0.6       ],
       [0.66666667, 0.        , 0.66666667, ..., 0.2       , 0.66666667,
        0.71428571],
       [0.8       , 0.66666667, 0.        , ..., 0.6       , 0.        ,
        0.6       ],
       ...,
       [0.83333333, 0.2       , 0.6       , ..., 0.        , 0.6       ,
        0.66666667],
       [0.8       , 0.66666667, 0.        , ..., 0.6       , 0.        ,
        0.6       ],
       [0.6       , 0.71428571, 0.6       , ..., 0.66666667, 0.6       ,
        0.        ]])

In [ ]:
# Weighted Hamming distance matrix

def weighted_hamming_distance_matrix(X_cat, weights=None):
    """
    Compute the weighted Hamming distance matrix for categorical variables.

    Parameters
    ----------
    X_cat : array-like of shape (n_samples, n_categorical_variables)
        Matrix containing ONLY the categorical variables.
        Each column must correspond to one original categorical variable.
        The columns must be in a fixed and known order, for example:

            ['Gender', 'Job', 'Area', 'CitySize', 'Investments']

        The function works with strings, integers, or categories, because
        Hamming distance only checks whether two values are equal or different.

    weights : array-like of shape (n_categorical_variables,), default=None
        Weight vector for the categorical variables.

        IMPORTANT:
        - weights[k] is the weight assigned to column X_cat[:, k]
        - therefore the order of `weights` MUST be exactly the same as the
          column order in `X_cat`

        Example:
        if

            X_cat columns = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']

        then

            weights = np.array([1, 2, 1, 1, 3])

        means:
            Gender      -> weight 1
            Job         -> weight 2
            Area        -> weight 1
            CitySize    -> weight 1
            Investments -> weight 3

        If weights=None, all variables receive the same weight.

    Returns
    -------
    D : ndarray of shape (n_samples, n_samples)
        Symmetric weighted Hamming distance matrix.
        Values are in [0, 1] because the weighted mismatch score is normalized
        by the sum of the weights.

    Notes
    -----
    The distance between two observations i and j is:

        d(i, j) = sum_k [ w_k * I(x_ik != x_jk) ] / sum_k w_k

    where:
        - I(.) is 1 if the two categorical values are different, 0 otherwise
        - w_k is the weight of variable k
    """
    X_cat = np.asarray(X_cat)
    n_samples, n_features = X_cat.shape

    if weights is None:
        weights = np.ones(n_features, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    if weights.shape[0] != n_features:
        raise ValueError(
            f"`weights` must have length {n_features}, "
            f"but got length {weights.shape[0]}."
        )

    if np.any(weights < 0):
        raise ValueError("`weights` must contain non-negative values.")

    weight_sum = weights.sum()
    if weight_sum == 0:
        raise ValueError("The sum of `weights` must be greater than 0.")

    D = np.zeros((n_samples, n_samples), dtype=float)

    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            mismatches = (X_cat[i] != X_cat[j]).astype(float)
            d_ij = np.sum(weights * mismatches) / weight_sum
            D[i, j] = d_ij
            D[j, i] = d_ij

    return D


In [ ]:
# Weighted Tanimoto distance matrix

def weighted_tanimoto_distance_matrix(X_bin, weights=None):
    """
    Compute the weighted Tanimoto/Jaccard distance matrix for one-hot encoded
    categorical variables.

    Parameters
    ----------
    X_bin : array-like of shape (n_samples, n_binary_features)
        Binary matrix containing ONLY the one-hot encoded categorical variables.
        Each column must be a dummy/binary variable (0/1 or False/True).

        This function is intended for data such as:

            X_cat_encoded = OneHotEncoder(...).fit_transform(...)

        The columns must be in the exact order produced by the encoder.
        To check the correct order, use:

            encoder.get_feature_names_out()

        Example:
            ['Gender_Male',
             'Job_Manager',
             'Job_Student',
             'Job_Retired',
             'Area_Urban',
             'Area_Rural',
             'CitySize_Large',
             'CitySize_Small',
             'Investments_Yes']

    weights : array-like of shape (n_binary_features,), default=None
        Weight vector for the one-hot encoded columns.

        IMPORTANT:
        - weights[k] is the weight assigned to column X_bin[:, k]
        - therefore the order of `weights` MUST be exactly the same as the
          order of the one-hot encoded columns in `X_bin`

        Example:
        if

            encoder.get_feature_names_out() =
            ['Gender_Male',
             'Job_Manager',
             'Job_Student',
             'Job_Retired',
             'Area_Urban',
             'Area_Rural',
             'CitySize_Large',
             'CitySize_Small',
             'Investments_Yes']

        then `weights` must have 9 elements in that same order.

        If weights=None, all dummy columns receive the same weight.

        Practical note:
        if you want to weight ORIGINAL categorical variables rather than
        individual dummy columns, then you should distribute the weight of
        each original variable across its corresponding dummy columns.

    Returns
    -------
    D : ndarray of shape (n_samples, n_samples)
        Symmetric weighted Tanimoto/Jaccard distance matrix.
        Values are in [0, 1].

    Notes
    -----
    For two binary vectors x and y, the weighted Tanimoto/Jaccard distance is:

        d(i, j) = 1 - intersection / union

    where:
        intersection = sum_k [ w_k * I(x_k = 1 and y_k = 1) ]
        union        = sum_k [ w_k * I(x_k = 1 or  y_k = 1) ]

    If union = 0, the distance is set to 0.
    """
    X_bin = np.asarray(X_bin)

    if not np.all(np.isin(X_bin, [0, 1, False, True])):
        raise ValueError(
            "`X_bin` must contain only binary values (0/1 or False/True)."
        )

    X_bin = X_bin.astype(int)
    n_samples, n_features = X_bin.shape

    if weights is None:
        weights = np.ones(n_features, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    if weights.shape[0] != n_features:
        raise ValueError(
            f"`weights` must have length {n_features}, "
            f"but got length {weights.shape[0]}."
        )

    if np.any(weights < 0):
        raise ValueError("`weights` must contain non-negative values.")

    if weights.sum() == 0:
        raise ValueError("The sum of `weights` must be greater than 0.")

    D = np.zeros((n_samples, n_samples), dtype=float)

    for i in range(n_samples):
        xi = X_bin[i]
        for j in range(i + 1, n_samples):
            xj = X_bin[j]

            intersection = np.sum(weights * ((xi == 1) & (xj == 1)))
            union = np.sum(weights * ((xi == 1) | (xj == 1)))

            d_ij = 0.0 if union == 0 else 1.0 - intersection / union
            D[i, j] = d_ij
            D[j, i] = d_ij

    return D

In [ ]:
# L1 distance for numerical features

def L1_distance(x, y):
    return np.sum(np.abs(x-y))

In [ ]:
# L2 distance for numerical features

def L2_distance(x, y):
    return np.sqrt(np.sum(np.square(x-y)))

In [ ]:
# Canberra distance for numerical features

def canberra_distance(x, y):
    num = np.abs(x - y)
    den = np.abs(x) + np.abs(y)
    fin = np.where( den != 0, num/den, 0)

    return np.sum(fin)

In [ ]:
def mixed_distance(x_num, x_cat, y_num, y_cat, alpha, num_dist, cat_dist):
    match num_dist:
        case "L1":
            d_num = L1_distance(x_num, y_num)
        case "L2":
            d_num = L2_distance(x_num, y_num)
        case "Canberra":
            d_num = canberra_distance(x_num, y_num)
    match cat_dist:
        case "Hamming":
            d_cat = hamming_distance(x_cat, y_cat)
        case "Tanimoto":
            d_cat = rogerstanimoto(x_cat, y_cat)
    return alpha*d_num + (1-alpha)*d_cat

def mixed_matrix_distance(X_num, X_cat, alpha, num_dist, cat_dist):
    m = X_cat.shape[0]
    D = np.zeros((m,m))
    for i in range(m):
        for j in range(i,m):
            D[i][j] = mixed_distance(X_num[i], X_cat[i], X_num[j], X_cat[j], alpha, num_dist, cat_dist)
            D[j][i] = D[i][j]  # Distance Matrix is symmetrcal
            
    return D

In [ ]:
D_L1_Hamming = mixed_matrix_distance(
    X_num, 
    X_cat_encoded, 
    alpha = 0.3, 
    num_dist = "L1",
    cat_dist = "Hamming"
)

D_L2_Hamming = mixed_matrix_distance(
    X_num, 
    X_cat_encoded, 
    alpha = 0.3, 
    num_dist = "L2",
    cat_dist = "Hamming"
)

D_Canberra_Hamming = mixed_matrix_distance(
    X_num, 
    X_cat_encoded, 
    alpha = 0.3, 
    num_dist = "Canberra",
    cat_dist = "Hamming"
)

D_L1_Tanimoto = mixed_matrix_distance(
    X_num, 
    X_cat_encoded, 
    alpha = 0.3, 
    num_dist = "L1",
    cat_dist = "Tanimoto"
)